# Gold Layer, Category Sales
**Source:** Silver Tables `silver.silver_orders` and `silver.silver_products`  
**Target:** Gold Delta Table `/delta/gold/category_sales` (Hive table `gold.gold_category_sales`)  
**Pattern:** Full Refresh (`CREATE OR REPLACE`)  
**Transformations:**
- Join `silver_orders` and `silver_products` on `product_id`
- Aggregate total sales per product category
  
**Layer:** Gold

---

In [2]:
import os, sys, logging
from pyspark.sql import SparkSession

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "silver_orders_db"   : "silver",
    "silver_orders_table": "silver_orders",
    "silver_products_db" : "silver",
    "silver_products_table": "silver_products",
    "gold_db"            : "gold",
    "gold_table"         : "gold_category_sales",
    "gold_path"          : "hdfs://master:8020/delta/gold/category_sales",
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Gold_Category_Sales")
        .master("spark://master:7077")
        .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.2.1")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .enableHiveSupport()
        .getOrCreate()
    )

def run():
    logger.info("Gold Category Sales pipeline started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    # Create gold database if not exists
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CONFIG['gold_db']}")

    # Build and execute the aggregation query
    query = f"""
        CREATE OR REPLACE TABLE {CONFIG['gold_db']}.{CONFIG['gold_table']}
        USING DELTA
        LOCATION '{CONFIG['gold_path']}'
        AS
        SELECT
            p.category AS product_category,
            SUM(o.total_amount) AS category_total_sales
        FROM {CONFIG['silver_orders_db']}.{CONFIG['silver_orders_table']} o
        JOIN {CONFIG['silver_products_db']}.{CONFIG['silver_products_table']} p
          ON o.product_id = p.product_id
        GROUP BY p.category
    """
    spark.sql(query)
    logger.info(f"Table {CONFIG['gold_db']}.{CONFIG['gold_table']} created/replaced.")

    # Verify and show sample
    count = spark.table(f"{CONFIG['gold_db']}.{CONFIG['gold_table']}").count()
    logger.info(f"SUCCESS: {count} rows in gold category sales table.")
    spark.table(f"{CONFIG['gold_db']}.{CONFIG['gold_table']}").show(10, truncate=False)

if __name__ == "__main__":
    run()

2026-03-19 13:45:05,284 [INFO] Gold Category Sales pipeline started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d14a7b10-f397-483c-bcdb-342dceda7d8b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (4446ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.1/delta-storage-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.1!delta-storage.jar (143ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (437ms)
:: resolution report :: resolve 2751ms

+----------------+--------------------+
|product_category|category_total_sales|
+----------------+--------------------+
|Home            |456502.7899999993   |
|Food            |441974.5000000002   |
|Sports          |510337.51999999967  |
|Electronics     |426449.8000000008   |
|Clothing        |519243.3299999996   |
|Automotive      |472281.32999999984  |
|Books           |429673.76000000053  |
|Garden          |483605.1099999999   |
|Beauty          |415598.3400000001   |
|Toys            |507586.1600000004   |
+----------------+--------------------+

